# 第 2 课 B：从数组到张量，再换一把数值刻度尺

**核心问题：同样的两条曲线，怎样准备成 PyTorch 可以处理的数据？**

你已经答对：4 条曲线、每条 7 个点，shape 为 `(4, 7)`，ndim 为 2，size 为 28。我们保留“每行一个样本、每列一个时间点”的约定，继续往前。

今天先完成两件事：

1. 将 NumPy 数组转成 PyTorch 张量，认识 `shape`、`dtype` 和 `device`。
2. 用统一的刻度缩放频率，并把缩放后的数值还原成 kHz。

我搭好：**环境和示例数据、检查过的计算过程。  
你来做：**解释数据变化，修改一个频率值，预测并核对缩放结果。  
仍使用 HEUhuqiao，逐格运行。本节只在 CPU 上处理 10 个数，尚未开始训练。

## 1．沿用上一课的频率数组

这份 Notebook 自己创建数据，不依赖上一份 Notebook 内核中的变量。两条曲线都在 0、0.1、0.2、0.3、0.4 秒各记录一个频率，单位 kHz。

`import torch` 加载 PyTorch。库的常用名字是 PyTorch，Python 导入名是 `torch`。

In [1]:
import numpy as np
import torch

batch_np = np.array([
    [8, 9, 12, 9, 8],
    [8, 9, 10, 9, 8],
], dtype=float)
print(batch_np)

[[ 8.  9. 12.  9.  8.]
 [ 8.  9. 10.  9.  8.]]


## 2．把数组交给 PyTorch

`torch.tensor(...)` 根据数据创建一个张量（Tensor）。本例会复制 NumPy 数据。

- `dtype=torch.float32`：每个数使用 32 位浮点格式，可以表示小数，是后面模型训练常用的起点；32 不是轴数或小数位数。
- `device="cpu"`：指定存放和计算的设备。这里显式使用 CPU。
- `x`：我们给这批输入起的短名字，没有特殊语法含义。

**先预测：**换了数据容器，是否会多出一条轴？曲线上的频率会不会自动归一化？

In [2]:
x = torch.tensor(batch_np, dtype=torch.float32, device="cpu")
print(x)
print("shape：", x.shape)
print("dtype：", x.dtype)
print("device：", x.device)

tensor([[ 8.,  9., 12.,  9.,  8.],
        [ 8.,  9., 10.,  9.,  8.]])
shape： torch.Size([2, 5])
dtype： torch.float32
device： cpu


这里输出 `torch.Size([2, 5])`，表示的尺寸仍是两行五列。`torch.Size` 是 PyTorch 用来表示形状的对象；方括号显示没有增加一条轴。

转换后的频率仍是原来的数。这次 8、9、10、12 都可以被精确表示；一般小数在 float32 中可能有舍入误差。

你可以先把 PyTorch 张量理解为“供 PyTorch 运算使用的多维数值容器”。张量可参与自动求导，也能放到支持的设备上计算，但**创建张量不等于已经训练了模型，也不会自动把 CPU 数据放上 GPU**。自动求导留到下一课讲。

同一个张量可以同时是“二维、float32、在 CPU 上”：形状、数值格式和设备描述的是三个不同方面。

## 3．原来的索引规则还能用吗？

`x[1, 2]` 仍取第二行第三列。输出 `tensor(10.)` 表示这个数仍由张量对象承载，不是一个新列表。

`x.ndim` 读取轴的数量。`x.numel()` 调用一个方法，得到总元素个数；`numel` 来自 number of elements。

注意：NumPy 的 `array.size` 是元素总数，而 PyTorch 中要用 `tensor.numel()` 求总数；`tensor.size()` 返回的是形状。不要只凭名字猜含义。

In [3]:
print("第二条曲线的第三个点：", x[1, 2])
print("ndim：", x.ndim)
print("总元素数：", x.numel())

第二条曲线的第三个点： tensor(10.)
ndim： 2
总元素数： 10


## 4．为什么要换数值刻度？

同一频率可以写成 8 kHz，也可以写成 8000 Hz。物理意义一样，交给程序的数值大小却不同。

后面优化器调整参数时，数值尺度会影响梯度大小和学习率的合适范围。选用一致的尺度能让训练设置更容易控制；但“归一化”不是训练一定成功的保证，也不意味着所有模型都必须把输入压到 0～1。

本例**事先约定参考范围为 8～12 kHz**，把它换成 0～1 的刻度。这个范围只服务于教学，不是论文参数，也不是所有鲸豚哨声的范围：

| 频率 | 在 8～12 kHz 这段范围中的位置 | 新刻度 |
|---|---|---|
| 8 kHz | 起点 | 0 |
| 10 kHz | 中点 | 0.5 |
| 12 kHz | 终点 | 1 |

实现这把刻度尺只需两步：

1. 减去下界，让 8 kHz 对应 0。
2. 除以范围宽度 `12 - 8`，让 12 kHz 对应 1。

缩放后的数是一个**没有物理单位的相对位置**。0.5 不是 0.5 kHz，也不是“发生概率为 50%”。

In [4]:
lower_khz = 8.0
upper_khz = 12.0

x_norm = (x - lower_khz) / (upper_khz - lower_khz)
print(x_norm)
print("缩放后的 shape：", x_norm.shape)

tensor([[0.0000, 0.2500, 1.0000, 0.2500, 0.0000],
        [0.0000, 0.2500, 0.5000, 0.2500, 0.0000]])
缩放后的 shape： torch.Size([2, 5])


这里减法和除法都逐个作用于张量中的数。得到新张量 `x_norm`，没有把 `x` 原地改掉。

括号让程序先计算上下界之差；`upper_khz` 必须大于 `lower_khz`，两者相等会造成除零。使用统一的上下界时，相同的频率总会对应相同的新刻度。

**停一下，自己检查：**第一条曲线的峰值原来是 12，第二条是 10。现在两个峰值是否仍能区分？shape 是否改变？

本节是一种固定范围的 min-max 缩放。不要对每条曲线分别临时选自己的最大最小值，否则同一个新刻度可能对应不同的绝对频率；恒频曲线还会遇到最大值等于最小值的问题。

## 5．能从新刻度恢复 kHz 吗？

把刚才的两步倒过来：先乘范围宽度，再加下界。下面的 `restored_khz` 是反变换结果，不是神经网络重建结果。

In [5]:
restored_khz = x_norm * (upper_khz - lower_khz) + lower_khz
print("还原为 kHz：")
print(restored_khz)
print("原始 kHz：")
print(x)

还原为 kHz：
tensor([[ 8.,  9., 12.,  9.,  8.],
        [ 8.,  9., 10.,  9.,  8.]])
原始 kHz：
tensor([[ 8.,  9., 12.,  9.,  8.],
        [ 8.,  9., 10.,  9.,  8.]])


因此以后保存模型时，也需要保留使用过的预处理规则和数值。本例需要记住下界 8、上界 12 以及原单位 kHz。

这里的边界是事先给定的。以后如果要从数据中估计最大最小值或均值方差，应只用**训练集**估计，再将同一套规则用于验证集和测试集；不能用测试集帮忙决定训练预处理。本节还没有划分数据集，这条规则会在训练流程课实际练习。

如果新频率超出了参考范围，结果可以小于 0 或大于 1；上述运算**不会自动截断**。是否需要截断取决于后续数据和模型设计，不能一看到大于 1 就判断计算错了。

## 6．你来改：11 kHz 会落在什么位置？

继续练习你已会的索引：只把第二条曲线第三个点改成 11 kHz，使用原来的 8～12 kHz 刻度。

先在下方学习记录中写预测：

- 11 kHz 缩放后是多少？写出你的计算过程。
- `shape` 和 `dtype` 会改变吗？

`x.clone()` 为这次练习复制一个张量，作用类似上一课数组的 `.copy()`。本节的张量没有梯度计算，先只关注副本可独立修改。

在注释处补**一行赋值**，随后运行整格。下面已给出缩放过程，不替你填目标位置。若未补赋值，输出仍是原数据，这不算完成练习。

In [6]:
edited = x.clone()

edited[1,2]=11.0# 在这里补一行：把第二条曲线第三个点改为 11 kHz。

edited_norm = (edited - lower_khz) / (upper_khz - lower_khz)
print("修改后对应点：", edited[1, 2])
print("缩放后对应点：", edited_norm[1, 2])
print("shape：", edited_norm.shape)
print("dtype：", edited_norm.dtype)
print("原始 x 对应点：", x[1, 2])

修改后对应点： tensor(11.)
缩放后对应点： tensor(0.7500)
shape： torch.Size([2, 5])
dtype： torch.float32
原始 x 对应点： tensor(10.)


### 最后一个迁移问题：不用新代码，先口头计算

如果下一条曲线出现 13 kHz，仍使用下界 8、上界 12，缩放后是多少？它是否一定在 0～1 内？

这个问题用来检验你是否理解刻度的来源，不要求你写额外的函数或检查程序。

## 我的学习记录（双击本格编辑）

- NumPy 数组转为张量后，形状是否改变：变成了torch.Size([2,5])
- `torch.float32` 与 `device="cpu"` 分别描述什么：张量里的数据类型；存放和运算工具
- 我补的赋值语句：edited[1,2]=11.0
- 11 kHz 的缩放结果与计算过程：0.75=(11-8)/(12-8)
- 实际输出是否符合预测：符合
- 修改并缩放后，shape / dtype 是否变化：都不变
- 13 kHz 的缩放结果，及为什么可以/不可以超出 0～1：1.25，但我觉得不应该超过，如果出现13那么13应该作为新的上界
- 想把缩放结果还原成 kHz，需要保留哪些信息：上下界
- 仍有疑问：暂时没有

## 先停在这里

把修改结果和解释发给我，或保存后让我读这份 Notebook。确认张量和缩放理解后，再补一小步“图像与 batch 的轴”，然后进入自动求导。今天不要求背训练代码。

保存时先运行修改过的代码格，再保存文件。下面在 PyCharm Terminal 中执行，并先核对暂存范围：

```powershell
git status
git add -- notebooks/02A_看懂数组形状与索引.ipynb notebooks/02B_张量与数值归一化.ipynb README.md 计划.md docs/学习记录.md
git diff --cached --stat
git commit -m "Learn tensors and frequency scaling"
```

若你同时改过 2A 的答案，请检查后单独选择该文件加入提交；这组命令不会自动包含它。需要上传时按 Git 手册执行 push。

可选参考：[PyTorch 张量入门](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)、[创建张量](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)、[预处理与数据泄漏](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)。官方在线文档可能更新，本课以 HEUhuqiao 中实际执行结果核对。